# Test: bliznji_odseki + agg_posek_sosedi

Notebook testira:
- `get_najblizje()` brez in s podano razdaljo
- `agg_odseki()` agregacijo za vrnjene ID-je
- Pravilnost izračunov (uteženo povprečje, vsote, robni primeri)

In [1]:
import sys
import subprocess
from pathlib import Path

# Locate project root via git, robust regardless of where Jupyter is launched from
project_root = Path(
    subprocess.check_output(["git", "rev-parse", "--show-toplevel"]).decode().strip()
)
SRC = project_root / "src" / "data_processing"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from bliznji_odseki import get_najblizje
from agg_posek_sosedi import agg_odseki, _load_all_odseki

import pandas as pd
import numpy as np

print(f"Project root: {project_root}")
print("Imports OK")

Project root: /home/tiln/Documents/BarkWatch-arnes_hackathon2026
Imports OK


## 1. get_najblizje — brez podane razdalje (default: top-10)

In [3]:
test_ids = ["31001", "10005", "50123"]

for oid in test_ids:
    try:
        result = get_najblizje(oid)
        print(f"get_najblizje({oid!r}) → {result}")
        assert result[0] == oid, "Prvi element mora biti odsek sam!"
        assert len(result) == 11, f"Pričakovali 11 (1 + 10), dobili {len(result)}"
    except KeyError as e:
        print(f"  SKIP: {e}")

get_najblizje('31001') → ['31001', '31002', '89B11', '31006A', '31006B', '31003', '31021', '89C17', '31022', '89C16A', '31007']
get_najblizje('10005') → ['10005', '10006', '10004', '10201', '10202', '10003', '10200', '10007A', '10008B', '10203', '10008A']
  SKIP: "Odsek '50123' not found."


## 2. get_najblizje — s podano razdaljo (max_km)

In [4]:
# Vzamemo prvi veljavni odsek iz podatkov za zanesljive teste
df_all = _load_all_odseki()
sample_ids = df_all["odsek"].astype(str).head(3).tolist()
print(f"Testni odseki: {sample_ids}")

Testni odseki: ['31001', '31002', '31003']


In [5]:
for oid in sample_ids:
    for km in [1.0, 5.0, 10.0]:
        result = get_najblizje(oid, max_km=km)
        print(f"get_najblizje({oid!r}, max_km={km}) → {len(result)} ID-jev (vključno s seboj)")
        assert result[0] == oid, "Prvi element mora biti odsek sam!"
        # Več km → vsaj toliko ali več sosedov
    r1 = get_najblizje(oid, max_km=1.0)
    r10 = get_najblizje(oid, max_km=10.0)
    assert len(r1) <= len(r10), "Manjši radij ne sme vrniti več sosedov kot večji!"
    print(f"  ✓ monotonija: |1km|={len(r1)} ≤ |10km|={len(r10)}")

get_najblizje('31001', max_km=1.0) → 8 ID-jev (vključno s seboj)
get_najblizje('31001', max_km=5.0) → 264 ID-jev (vključno s seboj)
get_najblizje('31001', max_km=10.0) → 1068 ID-jev (vključno s seboj)
  ✓ monotonija: |1km|=8 ≤ |10km|=1068
get_najblizje('31002', max_km=1.0) → 8 ID-jev (vključno s seboj)
get_najblizje('31002', max_km=5.0) → 263 ID-jev (vključno s seboj)
get_najblizje('31002', max_km=10.0) → 1078 ID-jev (vključno s seboj)
  ✓ monotonija: |1km|=8 ≤ |10km|=1078
get_najblizje('31003', max_km=1.0) → 13 ID-jev (vključno s seboj)
get_najblizje('31003', max_km=5.0) → 284 ID-jev (vključno s seboj)
get_najblizje('31003', max_km=10.0) → 1089 ID-jev (vključno s seboj)
  ✓ monotonija: |1km|=13 ≤ |10km|=1089


## 3. Robni primeri — napake

In [6]:
# Negativna razdalja → ValueError
try:
    get_najblizje(sample_ids[0], max_km=-1.0)
    print("FAIL: pričakovali ValueError")
except ValueError as e:
    print(f"✓ ValueError za max_km=-1: {e}")

# Neveljaven ID → KeyError
try:
    get_najblizje("NEOBSTOJI_999999")
    print("FAIL: pričakovali KeyError")
except KeyError as e:
    print(f"✓ KeyError za neveljaven ID: {e}")

✓ ValueError za max_km=-1: max_km mora biti nenegativen, dobili smo: -1.0
✓ KeyError za neveljaven ID: "Odsek 'NEOBSTOJI_999999' not found."


## 4. agg_odseki — agregacija za skupino odsekov

In [7]:
# Vzamemo sosede prvega testnega odseka
ids_default = get_najblizje(sample_ids[0])        # top-10
ids_radius  = get_najblizje(sample_ids[0], max_km=5.0)

agg_default = agg_odseki(ids_default)
agg_radius  = agg_odseki(ids_radius)

print(f"\n--- Agregacija top-10 ({agg_default['odseki_count']:.0f} odsekov) ---")
print(agg_default.to_string())

print(f"\n--- Agregacija radij 5 km ({agg_radius['odseki_count']:.0f} odsekov) ---")
print(agg_radius.to_string())


--- Agregacija top-10 (16 odsekov) ---
lzskdv11               26.895684
lzskdv11_m             37.101264
lzskdv21               35.595301
lzskdv21_m              4.064972
lzskdv30                0.177690
lzskdv30_m              0.000000
lzskdv34                0.046954
lzskdv34_m              0.013359
lzskdv39                0.000000
lzskdv39_m              0.000000
lzskdv41               24.977054
lzskdv41_m             22.183899
lzskdv50                1.364245
lzskdv50_m              0.321846
lzskdv60                6.264965
lzskdv60_m             11.211829
lzskdv70                3.293641
lzskdv70_m              4.327583
lzskdv80                0.086281
lzskdv80_m              1.892239
negovan                 1.560072
pompov                257.590000
lzigl              307954.000000
lzlst              179867.000000
lzsku              487821.000000
etigl               68671.000000
etlst               47508.000000
etsku              116179.000000
odseki_count           16.000000
pov

## 5. Preverjanje pravilnosti izračunov

In [8]:
# --- Test: lzsku ≈ lzigl + lzlst (skupna zaloga = iglavci + listavci) ---
for label, agg in [("top-10", agg_default), ("5km", agg_radius)]:
    if all(c in agg.index for c in ["lzsku", "lzigl", "lzlst"]):
        diff = abs(agg["lzsku"] - (agg["lzigl"] + agg["lzlst"]))
        ok = diff < 1e-6 or np.isnan(diff)  # toleranca za float
        print(f"✓ lzsku = lzigl + lzlst [{label}]: diff={diff:.6f} → {'OK' if ok else 'FAIL'}")

# --- Test: etsku ≈ etigl + etlst ---
for label, agg in [("top-10", agg_default), ("5km", agg_radius)]:
    if all(c in agg.index for c in ["etsku", "etigl", "etlst"]):
        diff = abs(agg["etsku"] - (agg["etigl"] + agg["etlst"]))
        ok = diff < 1e-6 or np.isnan(diff)
        print(f"✓ etsku = etigl + etlst [{label}]: diff={diff:.6f} → {'OK' if ok else 'FAIL'}")

✓ lzsku = lzigl + lzlst [top-10]: diff=0.000000 → OK
✓ lzsku = lzigl + lzlst [5km]: diff=0.000000 → OK
✓ etsku = etigl + etlst [top-10]: diff=0.000000 → OK
✓ etsku = etigl + etlst [5km]: diff=0.000000 → OK


In [9]:
# --- Test: uteženo povprečje je znotraj [min, max] vhodnih vrednosti ---
raw = df_all[df_all["odsek"].astype(str).isin(ids_default)].copy()

pct_cols = [c for c in [
    "lzskdv11", "lzskdv21", "lzskdv30", "lzskdv41", "lzskdv50"
] if c in raw.columns and c in agg_default.index]

for col in pct_cols:
    vals = pd.to_numeric(raw[col], errors="coerce").dropna()
    if vals.empty:
        continue
    agg_val = agg_default[col]
    in_range = (vals.min() - 1e-9) <= agg_val <= (vals.max() + 1e-9)
    print(f"  {col}: min={vals.min():.2f}, agg={agg_val:.2f}, max={vals.max():.2f} → {'✓' if in_range else 'FAIL'}")

  lzskdv11: min=3.17, agg=26.90, max=66.04 → ✓
  lzskdv21: min=0.00, agg=35.60, max=59.47 → ✓
  lzskdv30: min=0.00, agg=0.18, max=3.27 → ✓
  lzskdv41: min=7.52, agg=24.98, max=80.98 → ✓
  lzskdv50: min=0.00, agg=1.36, max=9.84 → ✓


In [10]:
# --- Test: prazen seznam → ValueError ---
try:
    agg_odseki([])
    print("FAIL: pričakovali ValueError")
except ValueError as e:
    print(f"✓ ValueError za prazen seznam: {e}")

# --- Test: neobstoječi ID-ji → ValueError ---
try:
    agg_odseki(["NEOBSTOJI_1", "NEOBSTOJI_2"])
    print("FAIL: pričakovali ValueError")
except ValueError as e:
    print(f"✓ ValueError za neobstoječe ID-je: {e}")

✓ ValueError za prazen seznam: odsek_ids list is empty.
✓ ValueError za neobstoječe ID-je: None of the given odsek IDs were found: ['NEOBSTOJI_1', 'NEOBSTOJI_2']


In [14]:
# --- Test: en sam odsek → agg = vrednosti tega odseka (za vsotne stolpce) ---
single_id = sample_ids[0]
agg_single = agg_odseki([single_id])

raw_rows = df_all[df_all["odsek"].astype(str) == single_id]
print(f"Odsek {single_id!r} se pojavi {len(raw_rows)}x v podatkih")
print(f"odseki_count iz agg_odseki: {agg_single['odseki_count']:.0f}")
assert int(agg_single["odseki_count"]) == len(raw_rows), \
    f"odseki_count ({agg_single['odseki_count']}) ne ustreza dejanskemu številu vrstic ({len(raw_rows)})"
print("✓ odseki_count ustreza dejanskemu številu vrstic")

def values_match(got, expected, tol=1e-6):
    if pd.isna(got) and pd.isna(expected):
        return True
    if pd.isna(got) or pd.isna(expected):
        return False
    return abs(float(got) - float(expected)) < tol

# Vrednostni test je smiseln samo ko je odsek enkrat v podatkih
if len(raw_rows) == 1:
    raw_single = raw_rows.iloc[0]
    for col in ["lzigl", "lzlst", "lzsku", "etigl", "etlst", "etsku"]:
        if col not in agg_single.index or col not in raw_single.index:
            print(f"  {col}: stolpec ni na voljo, preskočeno")
            continue
        expected = pd.to_numeric(raw_single[col], errors="coerce")
        got = agg_single[col]
        match = values_match(got, expected)
        print(f"  {col}: pričakovano={expected}, dobljeno={got} → {'✓' if match else 'FAIL'}")
else:
    print(f"  Vrednostni test preskočen — odsek se pojavi {len(raw_rows)}x (duplikati v podatkih)")

Odsek '31001' se pojavi 2x v podatkih
odseki_count iz agg_odseki: 2
✓ odseki_count ustreza dejanskemu številu vrstic
  Vrednostni test preskočen — odsek se pojavi 2x (duplikati v podatkih)
